# Load Data

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'year',
    'timestamp',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed.parquet',
                    columns=required_columns)

# Convert timestamp to datetime if it's not already
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Filter to remove all rows where timestamp is before or at January 27, 2009 = introduction of bounty stem
cutoff_date = pd.to_datetime('2009-01-27')
df = df[df['timestamp'] > cutoff_date]

# Recalculate userFEIsBounty after filtering
df['userFEIsBounty'] = df.groupby("userId")["isBounty"].transform(lambda x: x - x.mean())
df['logtimeSinceFirstActivityDays'] = np.log(1 + df['timeSinceFirstActivityDays'])
df['numQnAAT'] = df['numQuestionsAskedAT'] + df['numHelpProvidedAT']
df['lognumQnAAT'] = np.log(1 + df['numQnAAT'])

print(f"Data shape after filtering: {df.shape}")
print(f"Date range after filtering: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.info()

KeyboardInterrupt: 

In [2]:
df.to_parquet('../data/study_datasets/user_answers_bounty_processed_dateFiltered.parquet')

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'year',
    'timestamp',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed_dateFiltered.parquet',
                    columns=required_columns)

df['logtimeSinceFirstActivityDays'] = np.log(1 + df['timeSinceFirstActivityDays'])
df['numQnAAT'] = df['numQuestionsAskedAT'] + df['numHelpProvidedAT']
df['lognumQnAAT'] = np.log(1 + df['numQnAAT'])

print(f"Data shape after filtering: {df.shape}")
print(f"Date range after filtering: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.info()

Data shape after filtering: (32627382, 16)
Date range after filtering: 2009-01-27 00:00:02.727000 to 2025-03-31 23:59:12.307000
<class 'pandas.core.frame.DataFrame'>
Index: 32627382 entries, 0 to 32856257
Data columns (total 16 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   isBounty                             int64         
 1   userFEIsBounty                       float64       
 2   userId                               int64         
 3   year                                 int32         
 4   timestamp                            datetime64[ns]
 5   timeSinceFirstActivityDays           float64       
 6   logtimeSinceFirstActivityDays        float64       
 7   userFeLogTimeSinceFirstActivityDays  float32       
 8   numHelpProvidedAT                    int64         
 9   numQuestionsAskedAT                  int64         
 10  logNumHelpProvidedAT                 float64       
 11  logNumQuestionsA

# Get Descriptives

In [2]:
# Define columns for descriptive statistics
descriptive_columns = [
    'isBounty',
    'lognumQnAAT',
    'logtimeSinceFirstActivityDays'
]

def calculate_descriptives(data, columns):
    """Calculate descriptive statistics (μ, σ, min, max) for specified columns"""
    stats_dict = {}

    for col in columns:
        if col in data.columns:
            # Handle non-numeric columns by skipping them or converting
            if data[col].dtype in ['object', 'category']:
                continue

            stats_dict[col] = {
                'μ': data[col].mean(),
                'σ': data[col].std(),
                'min': data[col].min(),
                'max': data[col].max()
            }
        else:
            print(f"Warning: Column '{col}' not found in data")
    return pd.DataFrame(stats_dict).T

# Calculate overall descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===")
overall_stats = calculate_descriptives(df, descriptive_columns)
print(overall_stats.round(4))

=== DESCRIPTIVE STATISTICS ===
                                    μ       σ  min      max
isBounty                       0.0106  0.1025  0.0   1.0000
lognumQnAAT                    4.1821  2.2969  0.0  11.3513
logtimeSinceFirstActivityDays  5.6836  2.2468  0.0   8.7133


In [3]:
len(df)

32627382

# 1. Main Bounty Effect

In [7]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

sys.modules['stargazer.translators.statsmodels'].pd = pd


# Non-FE Models
formula1 = "isBounty ~ lognumQnAAT + logtimeSinceFirstActivityDays + C(year)"

# FE Models
formula2 = "userFEIsBounty ~ lognumQnAAT + logtimeSinceFirstActivityDays + C(year)"

model_names = ["1", "2"]
formulas = [formula1, formula2]

def fit_model(formula, model_name, data):
    """Fit a single model, save LaTeX results, and clean memory"""
    print(f"Fitting Model {model_name}...")
    print(f"Formula: {formula}")

    try:
        # Clear memory before fitting
        gc.collect()

        # Fit the model
        model = smf.ols(formula=formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['userId']}
        )

        print(f"✓ Model {model_name} fitted successfully")

        # Create individual Stargazer table for this model
        stargazer = Stargazer([model])
        stargazer.title(f"Model {model_name}: Effect of Receiving Answers on Providing Help")
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(True)

        # Generate LaTeX code
        latex_output = stargazer.render_latex()
        print(latex_output)

        # Clear the model from memory
        del model
        gc.collect()

        return True

    except MemoryError as e:
        print(f"✗ Memory error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False
    except Exception as e:
        print(f"✗ Error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False

successful_models = []
failed_models = []

for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"Processing {i+1}/{len(formulas)}")
    fit_model(formula, name, df)

Processing 1/2
Fitting Model 1...
Formula: isBounty ~ lognumQnAAT + logtimeSinceFirstActivityDays + C(year)
✗ Memory error fitting model 1: Unable to allocate 4.26 GiB for an array with shape (19, 30125493) and data type float64
Processing 2/2
Fitting Model 2...
Formula: userFEIsBounty ~ lognumQnAAT + logtimeSinceFirstActivityDays + C(year)
✓ Model 2 fitted successfully
\begin{table}[!htbp] \centering
  \caption{Model 2: Effect of Receiving Answers on Providing Help}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{1}{c}{\textit{Dependent variable: userFEIsBounty}} \
\cr \cline{2-2}
\\[-1.8ex] & (1) \\
\hline \\[-1.8ex]
 C(year)[T.2010] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2011] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2012] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2013] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2014] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2015] & -0.001$^{***}$ \\
& (0.000) \\
 C(year)[T.2016] & -0.001$^{***